# 🎯 [PRC01_M02] 계약서 텍스트 파싱, 불공정 조항 키워드 추출 및 조항별 정규화 전처리 모듈 구현 (훈련생 실습용)
### 훈련생 실습 워크북 (Google Colab / Jupyter Student Edition)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/edunlab/milicontent2/blob/main/uploads/PRC01/prc01_m02_student.ipynb)

---

## 📌 1. 프로젝트 및 미션 개요

| 항목 | 내용 |
| :--- | :--- |
| **과정 분류** | 조달·계약·국산화 (Level 3) |
| **미션 ID** | PRC01_M02 |
| **미션 주제** | 계약서 텍스트 파싱, 불공정 조항 키워드 추출 및 조항별 정규화 전처리 모듈 구현 |
| **소요 시간** | 약 2시간 |
| **환경 사양** | Python 3.10+, numpy, pandas, matplotlib, scipy, scikit-learn |

### 📖 프로젝트 목표
조달계획안과 계약서는 지체상금률, 선금, 보증, 하도급, 필수 조항 같은 항목이 규정에 맞는지 사람이 한 건씩 읽고 대조해야 해서 검토가 늦어지고 불공정 조항이나 납기 지체 위험 문구를 놓치기 쉽다. 본 프로젝트는 조달계획안과 계약서의 오류와 위험을 검증하는 도구를 만드는 과제로, 조달 업무의 병목과 위험 신호 기준, 조항 수정 대안, 계약 의뢰 근거와 변경 이력, 업무 효과 측정까지 한 흐름으로 구현한다. 교육용 가상 계약 자료로 계약 데이터 스키마 정의, 서식별 자료 정제와 조항 정규화, 룰 기반 규정 준수 판별, 자연어처리 기반 독소·납기 위험 문구 식별, 위험 점수와 신뢰도 엔진, 조항 하이라이트 대시보드와 오류 처리, 계약관·법무 검토 HITL 워크플로우, 1,000건 일괄 스크리닝 부하 시험, AAR 룰셋 재보정, 통합 패키지까지 단계별로 만든다. 최종 목표는 위험 계약을 근거와 함께 먼저 보여 주어 검토 시간을 줄이되, 규정 위반 여부의 판단과 계약 승인은 계약관·법무·승인권자가 하도록 지원하는 도구를 완성하는 것이다.

> **⚠️ [교육용 가상 데이터셋 원칙]**
> - 본 노트북의 모든 데이터(제원, 로그, 시나리오 등)는 보안 규정을 준수하여 **코드 내 난수 기반 가상 데이터로 생성**됩니다.
> - 실제 부대 위치, 작전 인가, 실제 전장 정보, 군사 기밀 데이터는 일체 포함되지 않습니다.

---

### 📁 프로젝트 표준 산출물 디렉터리 체계 (`docs/`, `src/`, `dist/`) 안내
> **💡 [실무 프로토타입 업무 도구 구축 가이드]**
> 본 프로젝트는 단순 일회성 데이터 분석 실습이 아닌, 실제 군 부서(군수사령부, 조달본부, 계약관리과 등)에 배포 가능한 **업무 자동화 소프트웨어 도구(Tool)**를 단계별로 조립·완성하는 과정입니다. 실습 중 산출물(Deliverables)로 요구되는 파일들은 아래 표준 디렉터리에 모듈별로 저장·누적됩니다.
> - **`docs/` (국방 행정 및 기술 명세서):** 업무 정의서(`prc01_as_is_to_be_spec.md`), 스키마 명세서(`schema_definition.md`), UAT 검증 보고서(`prc01_uat_feedback_report.md`), 7개 장 운영 매뉴얼(`docs/prc01_final_manual.md`)
> - **`src/` (업무 도구 소스코드):** 데이터 파서(`connectors/`), 규정 준수 룰 엔진(`engine/`), NLP 지능 모듈(`ai/`), 대시보드 UI(`ui/`), 결함 검증기(`utils/`), HITL 워크플로우(`workflow/`), 배치 러너(`pipeline/`)
> - **`dist/` (배포 패키지):** 원클릭 자동 설치 스크립트(`dist/prc01_package/setup.sh`), 설치 가이드(`README.md`)
> *(주피터 노트북 환경에서는 각 미션 셀 검증 후 `%%writefile src/...` 매직 명령어를 활용하여 해당 모듈 파일로 자동 추출할 수 있습니다.)*

---


In [ ]:
# ============================================================================
# [Google Colab 환경 설정] 필수 라이브러리 설치 및 임포트
# ============================================================================
!pip install -q numpy pandas matplotlib scipy scikit-learn

import os
import sys
import time
import math
import json
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# 그래프 영문 폰트 기본 설정 (환경 독립성 보장)
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['axes.unicode_minus'] = False

print(f"실행 환경 준비 완료 (Python {sys.version.split()[0]})")


---

## 🎯 미션 02. 계약서 텍스트 파싱, 불공정 조항 키워드 추출 및 조항별 정규화 전처리 모듈 구현
**소요시간**: 약 2.0시간

### 📋 미션 배경 및 수행 지침
#### 미션 배경 및 당위성

계약서 텍스트 파싱, 불공정 조항 키워드 추출 및 정규화 전처리 모듈을 구현한다. 실제 전술 현장에서는 단순 탐지에 그치지 않고 신속한 위협 판단과 신뢰도 높은 의사결정이 필수적이므로, **서식이 다른 계약 자료(CSV, 엑셀 내보내기, JSON, 계약서·공문 텍스트)를 하나의 표준 스키마로 바꾸고 불공정 조항 키워드를 추출하는 정규화 전처리 모듈을 구현한다.** 과업을 완수하여 오판을 방지하고 작전 대응력을 극대화해야 합니다.

이 문제를 해결하기 위해 본 미션에서는 **열 이름 별칭 사전으로 CSV, 탭 구분(엑셀 내보내기), JSON 세 형식을 표준 스키마로 바꾸고, 금액(억원·백만원·만원·원), 기간(일·주·개월·년), 지체상금률(%, 1/1000, 천분의), 선금 표기를 통일 수행하고 검증하는** 실습을 체계적으로 수행합니다.

학습자는 실습가이드에 제시된 총 5개 세부 활동(기본 기능 구현 ➔ 결과 지표 검증 ➔ 파라미터 튜닝 및 도전과제)을 순차적으로 진행하며 핵심 판정 기준을 달성합니다.

**이번 결과물:** `src/connectors/prc01_data_loader.py 및 스키마 정의서` 형태의 결과물을 완성한다.

### 🔍 평가 및 검증 기준 (PASS 조건)
#### 미션 판정

**평가 구분:** 필수

**핵심 평가영역:** 국방 서식/양식 변경에 대한 유연한 확장성

#### 미션 PASS 조건

- 정량 기준: 지원 파일 포맷(CSV, Excel, JSON 등) 파싱 성공률 100%


- 진행 기준: 데이터 로더 및 스키마 유효성 검사 모듈 완성


- 검증 방법: 다양한 형태의 샘플 군수 문서 대상 파싱 테스트


#### 채점자 판정 메모

> 단순 코드 실행 여부뿐만 아니라 출력된 수치와 근거 자료가 군 실무 요구사항에 부합하는지 면밀히 확인한다.



In [ ]:
"""서식이 서로 다른 가상 계약 자료(CSV, 엑셀 내보내기 TSV, JSON, 계약서·공문 텍스트)를 열 이름 별칭 사전과 정규식으로 표준 스키마로 바꾸고, 금액·기간·지체상금률·선금 표기와 조항 문장을 정규화하며 불공정 조항 키워드를 추출하는 전처리 모듈을 구현한다."""
import csv
import io
import json
import re
import pandas as pd

# ---------------------------------------------------------------------------
# [교육용가상자료] 서로 다른 형식의 원시 자료 (실제 계약 아님). 금액·기간·비율 표기가 일부러 제각각이다.
# ---------------------------------------------------------------------------
RAW_CSV = """계약번호,계약방식,계약금액,계약기간,지체상금률,선금
C-0101,일반경쟁,1.5억원,90일,0.15%,30%
C-0102,수의계약,"3,200만원",3개월,1.5/1000,20퍼센트
C-0103,제한경쟁,250백만원,1년,1.0‰,0 %
C-0104,지명경쟁,"1,200,000,000원",180일,천분의 2.0,10%
C-0105,수의계약,4800만원,45일,0.05%,60%
C-0106,일반경쟁,95백만원,12주,1.5/1000,0%
"""
RAW_TSV = "계약 번호\t계약 방식\t총액\t이행기간\t지체율\t선급금\n" \
          "C-0107\t일반경쟁\t0.8억원\t60일\t0.1%\t10%\n" \
          "C-0108\t수의계약\t290백만원\t2개월\t1.0/1000\t50퍼센트\n" \
          "C-0101\t일반경쟁\t150백만원\t3개월\t1.5/1000\t30%\n" \
          "C-0109\t제한경쟁\t5,000만원\t1년\t천분의 1.5\t0%\n" \
          "C-0110\t지명경쟁\t2.2억원\t240일\t0.2%\t20%\n"
RAW_JSON = json.dumps({"status": "OK", "data": [
    {"id": "C-0111", "method": "일반경쟁", "terms": {"amount": "75백만원", "period": {"value": 4, "unit": "month"}, "penalty": "1.5/1000", "advance": "10%"}},
    {"id": "C-0112", "method": "수의계약", "terms": {"amount": "3.4억원", "period": {"value": 20, "unit": "week"}, "penalty": "0.08%", "advance": "40퍼센트"}},
    {"id": "C-0113", "method": "제한경쟁", "terms": {"amount": "880만원", "period": {"value": 30, "unit": "day"}, "penalty": "2.0‰", "advance": "0%"}}]},
    ensure_ascii=False)
DOCS = [
    "계약서(가상)\n제1조(계약금액) 계약금액은 금 150,000,000원으로 한다.\n제2조(계약기간) 계약기간은 계약일부터 3개월로 한다.\n"
    "제3조(지체상금) 계약상대자가 납기를 지체한 경우 지체일수 1일마다 계약금액의 0.15%를 지체상금으로 부과한다.\n"
    "제4조(선금) 선금은 계약금액의 30퍼센트 이내에서 지급한다.\n제5조(검사) 발주기관은 납품 후 7일 이내에 검사한다.\n"
    "제6조(해지) 계약상대자는 언제든지 일방적으로 계약을 해지할 수 있으며 그에 따른 손해는 발주기관이 부담한다.",
    "수신: 계약담당관\n제목: 조달계획안 검토 요청\n1. 계약 개요\n  가. 계약금액 : 2.5억원\n  나. 계약기간 : 1년\n  다. 지체상금률 : 천분의 0.8\n  라. 선금 : 계약금액의 60 %\n"
    "2. 주요 조항\n  가. 납기는 자재 수급 사정에 따라 계약 상대자가 연장할 수 있다.\n  나. 하도급은 신고 없이 제한 없이 허용한다.",
    "[계약 조건표]\n계약금액: 320백만원\n계약기간: 120일\n지체상금률: 1.5/1000\n선금: 20%\n[조항 요약]\n"
    "(1) 지체상금은 면제 사유가 없는 한 부과한다.\n(2) 납품 즉시 검사에 합격한 것으로 본다.",
    "CONTRACT SUMMARY (virtual)\nContract Amount: KRW 95 million\nPeriod: 6 months\nLate Penalty: 1.0 permil\nAdvance Payment: 10 percent\n"
    "Clause: 지체상금은 전액 면제하며 발주기관은 이의를 제기하지 못한다.",
    "계약 1건 요약 - 금액 4,800만원 / 기간 45일 / 지체상금 1.0‰ / 선금 0% / 비고: 납기일 이후 1회에 한해 기한 없이 자동 연장되는 것으로 한다.",
]

# ---- 정규화 규칙(서식이 바뀌면 이 사전만 고치면 된다) ----
COL_ALIAS = {"contract_id": ["계약번호", "계약 번호", "id"], "method": ["계약방식", "계약 방식", "method"], "amount": ["계약금액", "총액", "amount"],
             "period": ["계약기간", "이행기간", "period"], "penalty": ["지체상금률", "지체율", "penalty"], "advance": ["선금", "선급금", "advance"]}
UNIT_DAYS = {"일": 1, "day": 1, "주": 7, "week": 7, "개월": 30, "month": 30, "months": 30, "년": 365, "year": 365}
KW = {"독소": ["면제", "일방적", "신고 없이", "제한 없이", "전액 부담", "포기", "누설", "지지 않는다"], "납기위험": ["자동 연장", "기한 없이", "연장할 수 있다", "정하지 않는다", "통보한다"]}
# 서식별 추출 규칙(비정형 서식 보완 시 여기에 정규식을 한 줄 추가한다)
DOC_PAT = {"amount": [r"(?:계약금액|금액|Contract Amount)\s*(?:은|는)?\s*[:：]?\s*(?:금\s*|KRW\s*)?([\d,\.]+\s*(?:억원|백만원|만원|million|원))"],
           "period": [r"(?:계약기간|기간|Period)\s*(?:은|는)?\s*[:：]?\s*(?:계약일부터\s*)?(\d+\s*(?:일|개월|년|months?|days?))"],
           "advance": [r"(?:선금|Advance Payment)[^\n/]*?([\d.]+\s*(?:%|퍼센트|percent))"]}
PEN_PAT = [r"천분의\s*([\d.]+)", r"([\d.]+)\s*%", r"([\d.]+)\s*(?:/1000|‰|permil)"]
CLAUSE_PAT = [r"^제\d+조\([^)]*\)\s*(.+)$", r"^\s*[가-라]\.\s*(.+다\.)$", r"^\(\d+\)\s*(.+)$", r"^Clause:\s*(.+)$", r"비고:\s*(.+)$"]


def norm_amount(s):
    """금액 표기를 백만원 단위 숫자로 통일: 1.5억원, 250백만원, 3,200만원, 1,200,000,000원, KRW 95 million."""
    # [TODO : 학생 실습 2] 계약서 텍스트 파싱, 불공정 조항 키워드 추출 및 조항별 정규화 전처리 모듈 구현 관련 핵심 로직을 구현하세요.
    # 힌트: 상단 주석 및 데이터 스키마/지침을 참고하여 작성합니다.
    # <<< 여기에 코드를 작성하세요 >>>
    pass  # 작성 후 pass를 제거하세요.
def norm_period(v):
    # [TODO : 학생 실습 2] 계약서 텍스트 파싱, 불공정 조항 키워드 추출 및 조항별 정규화 전처리 모듈 구현 관련 핵심 로직을 구현하세요.
    # 힌트: 상단 주석 및 데이터 스키마/지침을 참고하여 작성합니다.
    # <<< 여기에 코드를 작성하세요 >>>
    pass  # 작성 후 pass를 제거하세요.
def norm_rate(s, per_mille=False):
    """비율 표기를 통일: 지체상금률은 1/1000(천분율) 단위, 선금은 % 단위."""
    s = str(s).strip()
    for pat, k in [(r"^천분의\s*([\d.]+)$", 1), (r"^([\d.]+)\s*(?:/1000|‰|permil)$", 1), (r"^([\d.]+)\s*(?:%|퍼센트|percent)$", 10 if per_mille else 1)]:
        m = re.match(pat, s)
        if m:
            return round(float(m.group(1)) * k, 3)
    raise ValueError(f"비율 해석 불가: {s}")


def pick(row, key):
    for k in COL_ALIAS[key]:
        if k in row:
            return row[k]
    raise KeyError(f"필수 열 없음: {key}")


def to_std(row, src):
    return {"contract_id": str(pick(row, "contract_id")).strip(), "method": str(pick(row, "method")).strip(), "amount_m": round(norm_amount(pick(row, "amount")), 2),
            "period_days": norm_period(pick(row, "period")), "penalty_permil": norm_rate(pick(row, "penalty"), True), "advance_pct": norm_rate(pick(row, "advance")), "source": src}


def load(text, fmt):
    """형식별 로더: csv / tsv(엑셀 내보내기) / json. 행마다 성공·실패를 따로 센다."""
    if fmt == "json":
        rows = [{"id": r["id"], "method": r["method"], "amount": r["terms"]["amount"], "period": r["terms"]["period"], "penalty": r["terms"]["penalty"],
                 "advance": r["terms"]["advance"]} for r in json.loads(text)["data"]]
    else:
        rows = list(csv.DictReader(io.StringIO(text), delimiter="\t" if fmt == "tsv" else ","))
    ok, fail = [], []
    for r in rows:
        try:
            ok.append(to_std(r, fmt))
        except (KeyError, ValueError) as e:
            fail.append((r, str(e)))
    return ok, fail


def norm_clause(s):
    """조항 문장 정규화: 띄어쓰기·호칭 통일, 번호 제거."""
    s = re.sub(r"^\s*(?:제\d+조\([^)]*\)|\(\d+\)|[가-라]\.)\s*", "", s)
    s = s.replace("계약 상대자", "계약상대자").replace("발주 기관", "발주기관").replace("계약담당관", "발주기관")
    return re.sub(r"\s+", " ", s).strip()


def parse_doc(text):
    """계약서·공문 텍스트 1건에서 금액·기간·지체상금률·선금과 조항 문장을 추출한다."""
    out = {}
    for key, pats in DOC_PAT.items():
        for p in pats:
            m = re.search(p, text)
            if m:
                out[key] = m.group(1); break
        else:
            raise ValueError(f"필수 항목({key}) 추출 실패")
    pen = None
    for line in re.split(r"\n| / ", text):
        if re.search(r"지체상금|Late Penalty", line):
            for p in PEN_PAT:
                m = re.search(p, line)
                if m:
                    unit = "%" if p == PEN_PAT[1] else ""
                    pen = (m.group(1) + unit) if unit else (("천분의 " + m.group(1)) if p == PEN_PAT[0] else m.group(0)); break
        if pen:
            break
    if not pen:
        raise ValueError("필수 항목(지체상금률) 추출 실패")
    clauses = []
    for line in re.split(r"\n| / ", text):
        for p in CLAUSE_PAT:
            m = re.search(p, line.strip())
            if m:
                c = norm_clause(m.group(1) if p != CLAUSE_PAT[2] else line)
                if c.endswith(("다.", "한다")) or "다." in c:
                    clauses.append(c)
                break
    return {"amount_m": round(norm_amount(out["amount"].replace("KRW", "").strip()), 2), "period_days": norm_period(out["period"]),
            "penalty_permil": norm_rate(pen, True), "advance_pct": norm_rate(out["advance"].replace("percent", "퍼센트")), "clauses": clauses}


def extract_keywords(clause):
    return [f"{c}:{k}" for c, ks in KW.items() for k in ks if k in clause]


# 1) 형식별 파싱
results, total, success = [], 0, 0
for fmt, text in [("csv", RAW_CSV), ("tsv", RAW_TSV), ("json", RAW_JSON)]:
    ok, fail = load(text, fmt)
    results += ok; total += len(ok) + len(fail); success += len(ok)
    print(f"[{fmt.upper():4s}] {len(ok) + len(fail)}행 중 {len(ok)}행 성공" + (f", 실패 {[e for _, e in fail]}" if fail else ""))
docs, d_fail = [], 0
for i, t in enumerate(DOCS, 1):
    try:
        docs.append(parse_doc(t))
    except ValueError as e:
        d_fail += 1; print(f"문서 {i} 파싱 실패:", e)
total += len(DOCS); success += len(docs)
print(f"[계약서·공문 텍스트] {len(DOCS)}건 중 {len(docs)}건 성공")
rate = success / total
print(f"\n전체 파싱 성공률 {rate:.0%} ({success}/{total}) - 목표 100%: {'달성' if rate == 1 else '미달'}")

# 2) 정제 결과(표기 통일, 중복 계약 정리)
std = pd.DataFrame(results)
before = len(std)
dup = std[std.duplicated("contract_id", keep=False)].sort_values("contract_id")
same = dup.drop(columns="source").drop_duplicates().shape[0] == dup["contract_id"].nunique() if len(dup) else True
std = std.drop_duplicates("contract_id").sort_values("contract_id")
print(f"\n[정제된 계약 자료] {before}행 -> 중복 계약 정리 후 {len(std)}행 (중복 {before - len(std)}건: 표기는 달랐지만 정규화 후 값이 {'동일' if same else '불일치'})")
print(std.to_string(index=False))

# 3) 계약서 텍스트 파싱 결과와 불공정 조항 키워드
print("\n[계약서·공문 파싱 결과]")
for i, d in enumerate(docs, 1):
    print(f"  문서 {i}: 금액 {d['amount_m']:g}백만원, 기간 {d['period_days']}일, 지체상금률 {d['penalty_permil']:g}/1000, 선금 {d['advance_pct']:g}%, 조항 {len(d['clauses'])}개")
print("\n[조항별 정규화와 불공정·납기 위험 키워드]")
n_kw = 0
for i, d in enumerate(docs, 1):
    for c in d["clauses"]:
        kws = extract_keywords(c)
        if kws:
            n_kw += 1; print(f"  문서 {i}: {c[:38]}... -> {', '.join(kws)}")
print(f"키워드가 걸린 조항 {n_kw}개 (키워드는 후보 표시이며 실제 불공정 여부는 계약관이 판단한다. 예: 문서 3의 '면제'는 '면제 사유가 없는 한 부과'라는 정상 문장의 오탐 후보 -> 미션 4에서 문맥 모델로 보완)")

# 4) 서식 변경 대응 시험 ①: 열 이름이 바뀐 새 양식 -> 별칭 사전 추가
new_form = "계약 식별번호,방식,대금 총액,수행기간,지체상금 비율,선급\nC-0121,일반경쟁,120백만원,90일,1.5/1000,10%\n"
ok, fail = load(new_form, "csv")
print(f"\n새 양식(열 이름 변경) 1차 시도: 성공 {len(ok)} / 실패 {len(fail)} -> {fail[0][1] if fail else '-'}")
for k, alias in {"contract_id": "계약 식별번호", "method": "방식", "amount": "대금 총액", "period": "수행기간", "penalty": "지체상금 비율", "advance": "선급"}.items():
    COL_ALIAS[k].append(alias)
ok, fail = load(new_form, "csv")
print(f"별칭 사전 갱신 후: 성공 {len(ok)} / 실패 {len(fail)} (코드 수정 없이 설정만 변경)")

# 5) 서식 변경 대응 시험 ②: 비정형 공문의 표현이 달라진 경우 -> 정규식 한 줄 추가
new_doc = "수신: 계약담당관\n1. 대금 총액 : 1.2억원\n2. 계약기간 : 6개월\n3. 지체상금률 : 1.0/1000\n4. 선금 : 계약금액의 10%\n5. 조항: 납기일 이후 1회에 한해 기한 없이 자동 연장되는 것으로 한다."
try:
    parse_doc(new_doc); r1 = "성공"
except ValueError as e:
    r1 = f"실패({e})"
DOC_PAT["amount"].append(r"(?:대금 총액)\s*[:：]?\s*([\d,\.]+\s*(?:억원|백만원|만원|원))")
d = parse_doc(new_doc)
print(f"새 공문 표현('대금 총액') 1차 시도: {r1} -> 규칙 1줄 추가 후 금액 {d['amount_m']:g}백만원 추출 성공")


<details>
<summary><b>💡 [참고] 미션 02 예상 실행 결과 (Output)</b></summary>

```text
[CSV ] 6행 중 6행 성공
[TSV ] 5행 중 5행 성공
[JSON] 3행 중 3행 성공
[계약서·공문 텍스트] 5건 중 5건 성공

전체 파싱 성공률 100% (19/19) - 목표 100%: 달성

[정제된 계약 자료] 14행 -> 중복 계약 정리 후 13행 (중복 1건: 표기는 달랐지만 정규화 후 값이 동일)
contract_id method  amount_m  period_days  penalty_permil  advance_pct source
     C-0101   일반경쟁     150.0           90             1.5         30.0    csv
     C-0102   수의계약      32.0           90             1.5         20.0    csv
     C-0103   제한경쟁     250.0          365             1.0          0.0    csv
     C-0104   지명경쟁    1200.0          180             2.0         10.0    csv
     C-0105   수의계약      48.0           45             0.5         60.0    csv
     C-0106   일반경쟁      95.0           84             1.5          0.0    csv
     C-0107   일반경쟁      80.0           60             1.0         10.0    tsv
     C-0108   수의계약     290.0           60             1.0         50.0    tsv
     C-0109   제한경쟁      50.0          365             1.5          0.0    tsv
     C-0110   지명경쟁     220.0          240             2.0         20.0    tsv
     C-0111   일반경쟁      75.0          120             1.5         10.0   json
     C-0112   수의계약     340.0          140             0.8         40.0   json
     C-0113   제한경쟁       8.8           30             2.0          0.0   json

[계약서·공문 파싱 결과]
  문서 1: 금액 150백만원, 기간 90일, 지체상금률 1.5/1000, 선금 30%, 조항 6개
  문서 2: 금액 250백만원, 기간 365일, 지체상금률 0.8/1000, 선금 60%, 조항 2개
  문서 3: 금액 320백만원, 기간 120일, 지체상금률 1.5/1000, 선금 20%, 조항 2개
  문서 4: 금액 95백만원, 기간 180일, 지체상금률 1/1000, 선금 10%, 조항 1개
  문서 5: 금액 48백만원, 기간 45일, 지체상금률 1/1000, 선금 0%, 조항 1개

[조항별 정규화와 불공정·납기 위험 키워드]
  문서 1: 계약상대자는 언제든지 일방적으로 계약을 해지할 수 있으며 그에 따른 ... -> 독소:일방적
  문서 2: 납기는 자재 수급 사정에 따라 계약상대자가 연장할 수 있다.... -> 납기위험:연장할 수 있다
  문서 2: 하도급은 신고 없이 제한 없이 허용한다.... -> 독소:신고 없이, 독소:제한 없이
  문서 3: 지체상금은 면제 사유가 없는 한 부과한다.... -> 독소:면제
  문서 4: 지체상금은 전액 면제하며 발주기관은 이의를 제기하지 못한다.... -> 독소:면제
  문서 5: 납기일 이후 1회에 한해 기한 없이 자동 연장되는 것으로 한다.... -> 납기위험:자동 연장, 납기위험:기한 없이
키워드가 걸린 조항 6개 (키워드는 후보 표시이며 실제 불공정 여부는 계약관이 판단한다. 예: 문서 3의 '면제'는 '면제 사유가 없는 한 부과'라는 정상 문장의 오탐 후보 -> 미션 4에서 문맥 모델로 보완)

새 양식(열 이름 변경) 1차 시도: 성공 0 / 실패 1 -> '필수 열 없음: contract_id'
... (이하 생략)
```
</details>
